In [68]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd
from collections import defaultdict

In [69]:
raw_dir = Path("../data/raw/artists")

artists_files = list(raw_dir.glob("*.json"))
artists_files

[WindowsPath('../data/raw/artists/153c9281-268f-4cf3-8938-f5a4593e5df4.json'),
 WindowsPath('../data/raw/artists/5b11f4ce-a62d-471e-81fc-a69a8278c7da.json'),
 WindowsPath('../data/raw/artists/7527f6c2-d762-4b88-b5e2-9244f1e34c46.json'),
 WindowsPath('../data/raw/artists/83b9cbe7-9857-49e2-ab8e-b57b01038103.json'),
 WindowsPath('../data/raw/artists/ba0d6274-db14-4ef5-b28d-657ebde1a396.json'),
 WindowsPath('../data/raw/artists/ba853904-ae25-4ebb-89d6-c44cfbd71bd2.json')]

In [70]:
for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)
    print(file.name, "->", artist.get("name"))

153c9281-268f-4cf3-8938-f5a4593e5df4.json -> Soundgarden
5b11f4ce-a62d-471e-81fc-a69a8278c7da.json -> Nirvana
7527f6c2-d762-4b88-b5e2-9244f1e34c46.json -> Deftones
83b9cbe7-9857-49e2-ab8e-b57b01038103.json -> Pearl Jam
ba0d6274-db14-4ef5-b28d-657ebde1a396.json -> The Smashing Pumpkins
ba853904-ae25-4ebb-89d6-c44cfbd71bd2.json -> Blur


In [71]:
with open(artists_files[0], "r", encoding="utf-8") as f:
    artist = json.load(f)

artist.keys()

dict_keys(['life-span', 'area', 'ipis', 'type-id', 'isnis', 'sort-name', 'release-groups', 'begin-area', 'end-area', 'country', 'disambiguation', 'relations', 'id', 'tags', 'name', 'gender-id', 'gender', 'type'])

In [72]:
relations = artist.get("relations", [])

print("Artist:", artist.get("name"))
print("Total relations:", len(relations))

for r in relations[:5]:
    print(r.get("type"), "-", r.get("target-type"))

Artist: Soundgarden
Total relations: 13
member of band - artist
member of band - artist
member of band - artist
member of band - artist
member of band - artist


In [73]:
relation_type_counts = Counter()
target_type_counts = Counter()

for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)

    for rel in artist.get("relations", []):
        relation_type_counts[rel.get("type")] += 1
        target_type_counts[rel.get("target-type")] += 1

relation_type_counts

Counter({'tribute': 71,
         'member of band': 62,
         'instrumental supporting musician': 7,
         'named after artist': 1,
         'vocal supporting musician': 1,
         'artist rename': 1})

In [74]:
relationship_summary = (
    pd.DataFrame(relation_type_counts.items(), columns=["relationship_type", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

relationship_summary

,relationship_type,count
0,tribute,71
1,member of band,62
2,instrumental supporting musician,7
3,named after artist,1
4,vocal supporting musician,1
5,artist rename,1


In [75]:
target_summary = (
    pd.DataFrame(target_type_counts.items(), columns=["target_type", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

target_summary

,target_type,count
0,artist,143


In [76]:
tag_counts = Counter()

for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)

    for tag in artist.get("tags", []):
        tag_name = tag.get("name")
        if tag_name:
            tag_counts[tag_name] += 1

tag_counts

Counter({'alternative rock': 6,
         'grunge': 5,
         'rock': 4,
         'alternative metal': 3,
         'usa': 3,
         'shoegaze': 3,
         'american': 2,
         'hard rock': 2,
         'sludge metal': 2,
         'acoustic rock': 2,
         'estados unidos': 2,
         'noise rock': 2,
         'seattle': 2,
         'art rock': 2,
         'experimental rock': 2,
         'indie rock': 2,
         'post-grunge': 2,
         'neo-psychedelia': 2,
         'psychedelic rock': 1,
         '90s': 1,
         'pigfuck': 1,
         'post-hardcore': 1,
         'punk': 1,
         'punk rock': 1,
         'united states': 1,
         'heavy metal': 1,
         'metal': 1,
         'nu metal': 1,
         'bridge': 1,
         'christmas singles': 1,
         'classic rock': 1,
         'funk rock': 1,
         'pop rock': 1,
         'rock and indie': 1,
         'dream pop': 1,
         'progressive rock': 1,
         'spacegrunge': 1,
         'symphonic rock': 1,

In [77]:
tag_summary = (
    pd.DataFrame(tag_counts.items(), columns=["tag", "artist_count"])
    .sort_values("artist_count", ascending=False)
    .reset_index(drop=True)
)

tag_summary

,tag,artist_count
0,alternative rock,6
1,grunge,5
2,rock,4
3,usa,3
4,alternative metal,3
5,shoegaze,3
6,sludge metal,2
7,hard rock,2
8,acoustic rock,2
9,noise rock,2


In [78]:
tag_rows = []

for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)

    artist_name = artist.get("name")

    for tag in artist.get("tags", []):
        tag_rows.append({
            "artist_name": artist_name,
            "tag": tag.get("name"),
            "tag_count": tag.get("count")
        })

tag_df = pd.DataFrame(tag_rows)
tag_df.sort_values(["artist_name", "tag_count"], ascending=[True, False])

,artist_name,tag,tag_count
59,Blur,alternative rock,19
64,Blur,britpop,16
67,Blur,indie rock,14
63,Blur,british,8
60,Blur,art rock,3
...,...,...,...
54,The Smashing Pumpkins,rock,3
56,The Smashing Pumpkins,spacegrunge,3
53,The Smashing Pumpkins,progressive rock,2
57,The Smashing Pumpkins,symphonic rock,1


In [79]:
for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)
    for rel in artist.get("relations", []):
        if rel.get("type") == "tribute":
            print(artist.get("name"), "->", rel.get("target-type"), "-", rel.get("artist", {}).get("name"))
            break

Soundgarden -> artist - Bad Motorfingers
Nirvana -> artist - Seattle Supersonics
Deftones -> artist - Koi No Yokan – A Deftones Tribute
Pearl Jam -> artist - Alive – A Tribute to Pearl Jam
The Smashing Pumpkins -> artist - The Infinite Sadness
Blur -> artist - Blurz - A Tribute to Blur


In [80]:
for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist = json.load(f)
    for rel in artist.get("relations", []):
        if rel.get("type") == "member of band":
            print(artist.get("name"), "-", rel.get("artist", {}).get("name"), "| begin:", rel.get("begin"), "| end:", rel.get("end"))

Soundgarden - Hiro Yamamoto | begin: 1984 | end: 1989
Soundgarden - Chris Cornell | begin: 1984 | end: 2017-05-18
Soundgarden - Chris Cornell | begin: 1984 | end: 2017-05-18
Soundgarden - Kim Thayil | begin: 1984 | end: None
Soundgarden - Matt Cameron | begin: 1986 | end: None
Soundgarden - Jason Everman | begin: 1989 | end: 1990
Soundgarden - Ben Shepherd | begin: 1990 | end: None
Nirvana - Aaron Burckhard | begin: 1987 | end: 1987
Nirvana - Dale Crover | begin: 1987 | end: 1988
Nirvana - Kurt Cobain | begin: 1987 | end: 1994-04-05
Nirvana - Kurt Cobain | begin: 1987 | end: 1994-04-05
Nirvana - Kurt Cobain | begin: 1987 | end: 1994-04-05
Nirvana - Kurt Cobain | begin: 1987 | end: 1994-04-05
Nirvana - Krist Novoselic | begin: 1987 | end: 1994-04-05
Nirvana - Krist Novoselic | begin: 1987 | end: 1994-04-05
Nirvana - Krist Novoselic | begin: 1987 | end: 1994-04-05
Nirvana - Chad Channing | begin: 1988-05 | end: 1990-06
Nirvana - Dave Foster | begin: 1988 | end: 1988
Nirvana - Jason Everm

In [81]:
with open("../data/raw/artists/5b11f4ce-a62d-471e-81fc-a69a8278c7da.json", "r", encoding="utf-8") as f:  
    nirvana = json.load(f)

for rel in nirvana.get("relations", []):
    if rel.get("type") == "member of band" and rel.get("artist", {}).get("name") == "Kurt Cobain":
        print(rel)

{'ended': True, 'attribute-values': {}, 'target-credit': '', 'target-type': 'artist', 'source-credit': '', 'end': '1994-04-05', 'type': 'member of band', 'attributes': ['original'], 'type-id': '5be4c609-9afa-4ea0-910b-12ffb71e3821', 'direction': 'backward', 'begin': '1987', 'artist': {'sort-name': 'Cobain, Kurt', 'id': '3c9b356d-d1db-4f4e-a7a3-2d00c1d70255', 'country': 'US', 'name': 'Kurt Cobain', 'disambiguation': 'lead singer and guitarist for Nirvana', 'type': 'Person', 'type-id': 'b6e035f4-3ce9-331c-97df-83397230b0df'}, 'attribute-ids': {'original': '4fd3b255-a7d7-4424-9a63-40fa543b601c'}}
{'attribute-ids': {'original': '4fd3b255-a7d7-4424-9a63-40fa543b601c', 'lead vocals': '8e2a3255-87c2-4809-a174-98cb3704f1a5'}, 'artist': {'type': 'Person', 'type-id': 'b6e035f4-3ce9-331c-97df-83397230b0df', 'disambiguation': 'lead singer and guitarist for Nirvana', 'name': 'Kurt Cobain', 'country': 'US', 'id': '3c9b356d-d1db-4f4e-a7a3-2d00c1d70255', 'sort-name': 'Cobain, Kurt'}, 'begin': '1987', 

In [82]:
member_relations = defaultdict(lambda: {"begin": None, "end": None, "roles": set()})

with open("../data/raw/artists/5b11f4ce-a62d-471e-81fc-a69a8278c7da.json", "r", encoding="utf-8") as f:
    nirvana = json.load(f)

for rel in nirvana.get("relations", []):
    if rel.get("type") == "member of band":
        person_name = rel["artist"]["name"]
        key = (person_name, rel.get("begin"), rel.get("end"))

        member_relations[key]["begin"] = rel.get("begin")
        member_relations[key]["end"] = rel.get("end")
        
        attributes = rel.get("attributes", [])
        member_relations[key]["roles"].update(attributes)
        
member_relations

defaultdict(<function __main__.<lambda>()>,
            {('Aaron Burckhard', '1987', '1987'): {'begin': '1987',
              'end': '1987',
              'roles': {'drums (drum set)', 'original'}},
             ('Dale Crover', '1987', '1988'): {'begin': '1987',
              'end': '1988',
              'roles': {'drums (drum set)'}},
             ('Kurt Cobain', '1987', '1994-04-05'): {'begin': '1987',
              'end': '1994-04-05',
              'roles': {'background vocals',
               'guitar',
               'lead vocals',
               'original'}},
             ('Krist Novoselic', '1987', '1994-04-05'): {'begin': '1987',
              'end': '1994-04-05',
              'roles': {'accordion',
               'bass guitar',
               'original',
               'other vocals'}},
             ('Chad Channing', '1988-05', '1990-06'): {'begin': '1988-05',
              'end': '1990-06',
              'roles': {'drums (drum set)'}},
             ('Dave Foster', '1988', '1

In [83]:
all_member_relations = []

for file in artists_files:
    with open(file, "r", encoding="utf-8") as f:
        artist_data = json.load(f)
    
    artist_name = artist_data.get("name")
    grouped = defaultdict(lambda: {"begin": None, "end": None, "roles": set()})
    
    for rel in artist_data.get("relations", []):
        if rel.get("type") == "member of band":
            person_name = rel["artist"]["name"]
            key = (person_name, rel.get("begin"), rel.get("end"))
            grouped[key]["begin"] = rel.get("begin")
            grouped[key]["end"] = rel.get("end")
            grouped[key]["roles"].update(rel.get("attributes", []))
    
    for (person, begin, end), data in grouped.items():
        all_member_relations.append({
            "artist": artist_name,
            "person": person,
            "begin": data["begin"],
            "end": data["end"],
            "roles": list(data["roles"])
        })

member_df = pd.DataFrame(all_member_relations)
member_df

,artist,person,begin,end,roles
0,Soundgarden,Hiro Yamamoto,1984,1989,"[original, electric bass guitar]"
1,Soundgarden,Chris Cornell,1984,2017-05-18,"[original, lead vocals, guitar]"
2,Soundgarden,Kim Thayil,1984,NaN,"[original, guitar]"
3,Soundgarden,Matt Cameron,1986,NaN,[drums (drum set)]
4,Soundgarden,Jason Everman,1989,1990,[]
5,Soundgarden,Ben Shepherd,1990,NaN,[electric bass guitar]
6,Nirvana,Aaron Burckhard,1987,1987,"[original, drums (drum set)]"
7,Nirvana,Dale Crover,1987,1988,[drums (drum set)]
8,Nirvana,Kurt Cobain,1987,1994-04-05,"[background vocals, original, lead vocals, gui..."
9,Nirvana,Krist Novoselic,1987,1994-04-05,"[bass guitar, other vocals, original, accordion]"


## Phase 4A Relationship Inventory Findings

The raw MusicBrainz artist JSON contains relationship-rich metadata that is not present in `album_tracks.csv`.

The most important relationship type for Graph Schema v0 is `member of band`, with 62 raw relationships before deduplication. After grouping repeated entries by person, begin date, and end date, this relationship can be modeled as:

`(:Person)-[:MEMBER_OF {begin, end, roles}]->(:Artist)`

The `target_type` summary shows that all relationship targets are `artist` entities in MusicBrainz. However, MusicBrainz uses `artist` for both groups and people, so nested fields such as `artist.type` are needed to distinguish `Person` from `Group`.

Tags also provide useful genre/style overlap across the seed artists. `alternative rock` appears across all 6 artists, while `grunge` appears across 5 of 6 artists. This supports modeling tags as:

`(:Artist)-[:HAS_TAG]->(:Tag)`

Other relationship types are deferred or excluded from Graph Schema v0:
- `tribute` is excluded because it is noisy and less central to the retrieval comparison.
- `instrumental supporting musician` and `vocal supporting musician` are deferred because they may be useful later but are not core to v0.
- `named after artist` is excluded because it is too sparse.
- `artist rename` is deferred because it is useful for identity/history, but not necessary for the first retrieval-focused schema.